In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix, 
    precision_recall_fscore_support, roc_auc_score
)
from sklearn.base import BaseEstimator, ClassifierMixin
from scipy.optimize import minimize_scalar
import re

# Plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 7)

In [30]:
class ImprovedLayeredClassifier(BaseEstimator, ClassifierMixin):
    """
    Enhanced two-layer classifier with multiple improvements:
    - Class-weighted logistic regression
    - Optimizable soft routing weights
    - Better feature representation
    - Threshold calibration
    """
    
    def __init__(self, max_features=10000, ngram_range=(1, 2), 
                 routing_strategy='soft', class_weight='balanced', 
                 random_state=42, solver='lbfgs', max_iter=100, penalty='l2'):
        self.max_features = max_features
        self.ngram_range = ngram_range
        self.random_state = random_state
        self.routing_strategy = routing_strategy  # 'soft', 'hard', 'weighted', 'ensemble'
        self.class_weight = class_weight
        self.solver = solver
        self.max_iter = max_iter
        self.penalty = penalty
        
        # Layer 1: Group classifier (with class weights)
        self.layer1_classifier = LogisticRegression(
            max_iter=max_iter, 
            random_state=random_state,
            class_weight=class_weight,
            solver=solver,
            penalty=penalty
        )
        
        # Layer 2: Fine-grained classifiers (with class weights)
        self.layer2a_classifier = LogisticRegression(
            max_iter=max_iter, 
            random_state=random_state,
            class_weight=class_weight,
            solver=solver,
            penalty=penalty
        )  # For 1/2/3
        self.layer2b_classifier = LogisticRegression(
            max_iter=max_iter, 
            random_state=random_state,
            class_weight=class_weight,
            solver=solver,
            penalty=penalty
        )  # For 3/4/5
        
        # Vectorizers with enhanced n-grams
        self.layer1_vectorizer = TfidfVectorizer(
            max_features=max_features, 
            ngram_range=ngram_range,
            min_df=2,  # Ignore very rare terms
            max_df=0.95  # Ignore very common terms
        )
        self.layer2a_vectorizer = TfidfVectorizer(
            max_features=max_features, 
            ngram_range=ngram_range,
            min_df=2,
            max_df=0.95
        )
        self.layer2b_vectorizer = TfidfVectorizer(
            max_features=max_features, 
            ngram_range=ngram_range,
            min_df=2,
            max_df=0.95
        )
        
        # Routing weights (can be optimized)
        self.routing_weights = None
    
    def fit(self, X, y, X_val=None, y_val=None):
        """
        Train the improved layered classifier.
        
        Args:
            X: Series of text reviews
            y: Series of star ratings (1-5)
            X_val: Optional validation set for routing optimization
            y_val: Optional validation labels
        """
        if not isinstance(X, pd.Series):
            X = pd.Series(X)
        if not isinstance(y, pd.Series):
            y = pd.Series(y)
        
        print("\n" + "="*80)
        print("TRAINING IMPROVED LAYERED CLASSIFIER")
        print(f"Strategy: {self.routing_strategy.upper()} | Class Weight: {self.class_weight}")
        print("="*80)
        
        # --- LAYER 1: BINARY GROUP CLASSIFIER ---
        print("\n[LAYER 1] Training binary group classifier (1/2/3 vs 3/4/5)")
        print("-" * 80)
        
        group_labels_list = []
        X_list = []
        
        mask_123 = y <= 3
        X_123 = X[mask_123]
        X_list.append(X_123)
        group_labels_list.append(np.zeros(len(X_123), dtype=int))
        
        mask_345 = y >= 3
        X_345 = X[mask_345]
        X_list.append(X_345)
        group_labels_list.append(np.ones(len(X_345), dtype=int))
        
        X_layer1 = pd.concat(X_list, ignore_index=True)
        group_labels = np.concatenate(group_labels_list)
        
        X_layer1_tfidf = self.layer1_vectorizer.fit_transform(X_layer1)
        self.layer1_classifier.fit(X_layer1_tfidf, group_labels)
        
        layer1_acc = accuracy_score(
            group_labels, self.layer1_classifier.predict(X_layer1_tfidf)
        )
        print(f"  Training Accuracy: {layer1_acc:.4f}")
        print(f"  Samples: {len(X_layer1)} (Group 0: {(group_labels==0).sum()}, "
              f"Group 1: {(group_labels==1).sum()})")
        
        # --- LAYER 2A: FINE CLASSIFIER FOR 1/2/3 ---
        print("\n[LAYER 2A] Training fine classifier for 1/2/3 stars")
        print("-" * 80)
        
        mask_123_fine = y <= 3
        X_123_fine = X[mask_123_fine]
        y_123_fine = y[mask_123_fine]
        
        X_123_tfidf = self.layer2a_vectorizer.fit_transform(X_123_fine)
        self.layer2a_classifier.fit(X_123_tfidf, y_123_fine)
        
        layer2a_acc = accuracy_score(
            y_123_fine, self.layer2a_classifier.predict(X_123_tfidf)
        )
        print(f"  Training Accuracy: {layer2a_acc:.4f}")
        print(f"  Samples: {len(X_123_fine)} (1: {(y_123_fine==1).sum()}, "
              f"2: {(y_123_fine==2).sum()}, 3: {(y_123_fine==3).sum()})")
        
        # --- LAYER 2B: FINE CLASSIFIER FOR 3/4/5 ---
        print("\n[LAYER 2B] Training fine classifier for 3/4/5 stars")
        print("-" * 80)
        
        mask_345_fine = y >= 3
        X_345_fine = X[mask_345_fine]
        y_345_fine = y[mask_345_fine]
        
        X_345_tfidf = self.layer2b_vectorizer.fit_transform(X_345_fine)
        self.layer2b_classifier.fit(X_345_tfidf, y_345_fine)
        
        layer2b_acc = accuracy_score(
            y_345_fine, self.layer2b_classifier.predict(X_345_tfidf)
        )
        print(f"  Training Accuracy: {layer2b_acc:.4f}")
        print(f"  Samples: {len(X_345_fine)} (3: {(y_345_fine==3).sum()}, "
              f"4: {(y_345_fine==4).sum()}, 5: {(y_345_fine==5).sum()})")
        
        # Optional: Optimize routing weights on validation set
        if X_val is not None and y_val is not None:
            print("\n[OPTIMIZATION] Optimizing routing weights on validation set...")
            self._optimize_routing_weights(X_val, y_val)
        else:
            self.routing_weights = np.array([0.5, 0.5])  # Default equal weights
        
        print("\n" + "="*80)
        print("Training Complete")
        print("="*80 + "\n")
        
        return self
    
    def _optimize_routing_weights(self, X_val, y_val):
        """
        Optimize the soft routing weights using validation set.
        """
        def objective(w1):
            w = np.array([w1, 1 - w1])
            self.routing_weights = w
            preds = self.predict(X_val)
            return -f1_score(y_val, preds, average='weighted')
        
        result = minimize_scalar(objective, bounds=(0.2, 0.8), method='bounded')
        self.routing_weights = np.array([result.x, 1 - result.x])
        print(f"  Optimized weights: Group0={self.routing_weights[0]:.4f}, "
              f"Group1={self.routing_weights[1]:.4f}")
    
    def predict_proba(self, X):
        """
        Predict probabilities using the specified routing strategy.
        """
        if not isinstance(X, pd.Series):
            X = pd.Series(X)
        
        # Layer 1 predictions
        X_layer1_tfidf = self.layer1_vectorizer.transform(X)
        group_proba = self.layer1_classifier.predict_proba(X_layer1_tfidf)
        
        # Layer 2 predictions
        X_layer2a_tfidf = self.layer2a_vectorizer.transform(X)
        X_layer2b_tfidf = self.layer2b_vectorizer.transform(X)
        
        proba_123 = self.layer2a_classifier.predict_proba(X_layer2a_tfidf)
        proba_345 = self.layer2b_classifier.predict_proba(X_layer2b_tfidf)
        
        classes_123 = self.layer2a_classifier.classes_
        classes_345 = self.layer2b_classifier.classes_
        
        # Initialize combined probabilities
        combined_proba = np.zeros((len(X), 5))
        
        if self.routing_strategy == 'soft':
            # Soft routing: weighted combination using group probabilities
            weights = self.routing_weights if self.routing_weights is not None else np.array([0.5, 0.5])
            for i, class_label in enumerate(classes_123):
                combined_proba[:, class_label - 1] += weights[0] * proba_123[:, i]
            for i, class_label in enumerate(classes_345):
                combined_proba[:, class_label - 1] += weights[1] * proba_345[:, i]
        
        elif self.routing_strategy == 'dynamic':
            # Dynamic routing: use actual group probabilities as weights
            for i, class_label in enumerate(classes_123):
                combined_proba[:, class_label - 1] += group_proba[:, 0] * proba_123[:, i]
            for i, class_label in enumerate(classes_345):
                combined_proba[:, class_label - 1] += group_proba[:, 1] * proba_345[:, i]
        
        elif self.routing_strategy == 'hard':
            # Hard routing: pick one path based on group prediction
            group_preds = self.layer1_classifier.predict(X_layer1_tfidf)
            for i in range(len(X)):
                if group_preds[i] == 0:
                    for j, class_label in enumerate(classes_123):
                        combined_proba[i, class_label - 1] = proba_123[i, j]
                else:
                    for j, class_label in enumerate(classes_345):
                        combined_proba[i, class_label - 1] = proba_345[i, j]
        
        elif self.routing_strategy == 'ensemble':
            # Ensemble: average both paths equally
            for i, class_label in enumerate(classes_123):
                combined_proba[:, class_label - 1] += 0.5 * proba_123[:, i]
            for i, class_label in enumerate(classes_345):
                combined_proba[:, class_label - 1] += 0.5 * proba_345[:, i]
        
        # Normalize
        combined_proba = combined_proba / combined_proba.sum(axis=1, keepdims=True)
        
        return combined_proba
    
    def predict(self, X):
        proba = self.predict_proba(X)
        return np.argmax(proba, axis=1) + 1

In [31]:
# Load data
def load_data(folder_path='../Preprocessing-FeatureExtraction/cleaned-data'):
    csv_files = glob.glob(os.path.join(folder_path, '*.csv'))
   
    dfs = []
    for file in csv_files:
        df = pd.read_csv(file)
        dfs.append(df)
    
    data = pd.concat(dfs, ignore_index=True)
    data = data.dropna(subset=['stars', 'clean_text'])
    data['stars'] = data['stars'].astype(int)
    return data

data = load_data()

cherry_picked_examples = []
cherry_picked_idx = []
reviews_text = ["Horrible service and sushi was fishy and disgusting ... my sushi cook couldn't speak English and my whole experience was horrible ... go somewhere else where the sushi chefs are Japanese.",
                "Local small town hangout. The place had a weird smell of old rags used for wiping the bar down or something. It smells musty  from mold or mildew hiding out. The food is okay.",
                "We love this place.  It is kid friendly and they have a great selection of Crab Legs.",
                "Had the soup and salad lunch combo. Broccoli asiago soup is super yummy!! Love the homemade chips on the side."]

for text in reviews_text:
    pattern = re.escape(text)
    match = data.loc[data["text"].str.contains(pattern, regex=True, na=False)]
    cherry_picked_examples.append(match)
    cherry_picked_idx.extend(match.index.tolist()) 

data = data.drop(cherry_picked_idx)
X_pres = pd.concat(cherry_picked_examples)['clean_text']
print(X_pres)

train_size=400000
test_size=50000
val_size=50000
total_size = min(train_size + test_size + val_size, len(data))

print(f"Sampling {total_size} reviews...")

samples_per_class = min(total_size // data['stars'].nunique(), data['stars'].value_counts().min())
data_sample = (
    data.groupby('stars', group_keys=False)
        .apply(lambda x: x.sample(n=samples_per_class))
        .reset_index(drop=True)
)

print()
print(f"\nBalanced class distribution:")
print(data_sample['stars'].value_counts().sort_index())
print()

# Prepare features and labels
X = data_sample['clean_text'].reset_index(drop=True)
y = data_sample['stars'].astype(int).reset_index(drop=True)

# Manual stratified split
X_train, X_temp, y_train, y_temp = [], [], [], []
X_val, X_test, y_val, y_test = [], [], [], []

for star_class in sorted(y.unique()):
    mask = y == star_class
    X_class = X[mask].reset_index(drop=True)
    y_class = y[mask].reset_index(drop=True)
    
    n_class = len(y_class)
    n_train_c = max(15, int(n_class * train_size / total_size))
    n_val_c = max(5, int(n_class * val_size / total_size))
    n_test_c = max(5, n_class - n_train_c - n_val_c)
    
    indices = np.random.RandomState(42).permutation(n_class)
    
    X_train.append(X_class.iloc[indices[:n_train_c]])
    y_train.append(y_class.iloc[indices[:n_train_c]])
    
    val_end = n_train_c + n_val_c
    X_val.append(X_class.iloc[indices[n_train_c:val_end]])
    y_val.append(y_class.iloc[indices[n_train_c:val_end]])
    
    X_test.append(X_class.iloc[indices[val_end:val_end+n_test_c]])
    y_test.append(y_class.iloc[indices[val_end:val_end+n_test_c]])

X_train = pd.concat(X_train, ignore_index=True)
y_train = pd.concat(y_train, ignore_index=True)
X_val = pd.concat(X_val, ignore_index=True)
y_val = pd.concat(y_val, ignore_index=True)
X_test = pd.concat(X_test, ignore_index=True)
y_test = pd.concat(y_test, ignore_index=True)

print(f"\nTraining set: {len(X_train)} reviews | Validation: {len(X_val)} | Test: {len(X_test)}")

clf = ImprovedLayeredClassifier(
    max_features=10000,
    ngram_range=(1, 3),
    routing_strategy='hard',
    class_weight=None,
    random_state=42,
    solver='lbfgs',
    max_iter=300,
    penalty='l2'
)

min_df = 3
max_df = 0.9
max_features = 10000
ngram_range = (1,3)

clf.layer1_vectorizer = TfidfVectorizer(max_features=max_features, ngram_range=ngram_range,
                                            min_df=min_df, max_df=max_df)
clf.layer2a_vectorizer = TfidfVectorizer(max_features=max_features, ngram_range=ngram_range,
                                            min_df=min_df, max_df=max_df)
clf.layer2b_vectorizer = TfidfVectorizer(max_features=max_features, ngram_range=ngram_range,
                                            min_df=min_df, max_df=max_df)

# Set logistic regression Cs if given
c1 = 1.0
c2a = 1.0
c2b = 1.0
# Update classifier objects (before fit)
clf.layer1_classifier.C = c1
clf.layer2a_classifier.C = c2a
clf.layer2b_classifier.C = c2b

# Fit
clf.fit(X_train, y_train)

# Test
y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average='macro')

print(f"Test Accuracy: {acc:.4f}")
print(f"Test Macro-F1: {macro_f1:.4f}")

# Confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=[1, 2, 3, 4, 5])
# Normalized confusion matrix
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=[1,2,3,4,5], yticklabels=[1,2,3,4,5], ax=axes[0],
            cbar_kws={'label': 'Count'})
axes[0].set_title('Confusion Matrix - Raw Counts', fontsize=13, fontweight='bold')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# Normalized
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='RdYlGn',
            xticklabels=[1,2,3,4,5], yticklabels=[1,2,3,4,5], ax=axes[1],
            cbar_kws={'label': 'Percentage'}, vmin=0, vmax=1)
axes[1].set_title('Confusion Matrix - Normalized (% by true class)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

In [ ]:
for test in X_pres:
    print(test)
    guess = clf.predict(test)
    print(guess)

like single cheese burger fries nothing special burger salted expecting lot price
[2]
got stop roast pork italiano broccoli rabe provolone sharp oregon ave near packer ave stadiums park middle streeteveryone else tipuse bathroom get facilities
[4]
tourist trap dont waste hour vacation everyone recommends place taking locals experiencing definitely overrated gumbo wayyyy save time money hit different spot
[1]
hate giving one star shells eaten years weekly sometimes twice week adding tampa location like weekend plus tip average visit food good individual server good wife alz suffers memory lapses last time rather scooping peanuts paper tray scooped peanuts carried scoop table empty paper tray hostess came asked started laughing done said never saw anyone promptly gave scoop back someone laugh embarrass really upset hurt feelings treat guests important serve case never want return annual left shells go sea hags leverocks local seafood place sorry
[1]
